# 06 · Opsiyon Karar Hunisi

Dört adımda bir opsiyon fikrini veriyle sınar:

1. **Hisse:** Reddit (ApeWisdom) ve StockTwits'te en çok konuşulanlar, opsiyon likiditesine göre süzülür.
2. **Yön:** Söz (StockTwits boğa oranı), para (put/call oranı, çarpıklık) ve fiyat (hareketli ortalamalar) yan yana.
3. **Vade ve strike:** 7–45 gün arası vadeler, beklenen hareket, bilanço tarihi; 0,20–0,30 deltalık kısa bacak adayları.
4. **Yapı:** IV rank (MarketChameleon'dan elle girilir) ya da IV / gerçekleşen volatilite oranına göre opsiyonun pahalı mı ucuz mu olduğu; tek bacak, debit, kredi ve çok bacaklı yapıların gerçek primlerle karşılaştırması.

**Kullanım:** Ayarları gözden geçir, sonra **Çalışma zamanı > Tümünü çalıştır** (Ctrl+F9). API anahtarı gerekmez, maliyeti yoktur.

> ⚠️ Not defteri emir önermez; veriyi ve senaryoları gösterir. Hesaplar vade sonu ve teorik orta fiyat üzerinedir; gerçek dolum, komisyon ve erken kullanım farklılık yaratır. **Öğrenme amaçlıdır, yatırım tavsiyesi değildir.**

In [ ]:
#@title 🎯 Ayarlar ve kuralların
secilen_hisse = "" #@param {type:"string"}
#@markdown Boş bırakırsan Adım 1'deki en üst sıradaki likit hisse seçilir.
yon_secimi = "otomatik" #@param ["otomatik", "Yükseliş", "Düşüş", "Nötr"]
#@markdown Sinyallere katılmıyorsan yönü kendin seçebilirsin.
vade_secimi = "" #@param {type:"string"}
#@markdown Boş: 30 güne en yakın ve bilançoyu kapsamayan vade. Elle yazmak için biçim: 2026-12-19
#@markdown ---
#@markdown **Kuralların**
MIN_GUN = 7 #@param {type:"integer"}
MAX_GUN = 45 #@param {type:"integer"}
MIN_DELTA = 0.20 #@param {type:"number"}
MAX_DELTA = 0.30 #@param {type:"number"}
MAX_GENISLIK_YUZDE = 4.0 #@param {type:"number"}
#@markdown Spread genişliği üst sınırı, hisse fiyatının yüzdesi olarak. Bu sınırdaki en geniş strike seçilir.
MIN_ACIK_POZISYON = 100 #@param {type:"integer"}
MAX_MAKAS_YUZDE = 10 #@param {type:"number"}
#@markdown Likidite: kontrat başına en az açık pozisyon ve alış-satış makasının orta fiyata oranı.
#@markdown ---
REDDIT_FILTRESI = "all-stocks" #@param ["all-stocks", "wallstreetbets", "options", "stocks", "investing"]
ADAY_SAYISI = 15 #@param {type:"integer"}
IVR_PAHALI = 50 #@param {type:"integer"}
IVR_UCUZ = 25 #@param {type:"integer"}
#@markdown IV rank eşikleri: bu değerin üstü pahalı, altı ucuz sayılır.
iv_kaydet = False #@param {type:"boolean"}
#@markdown İşaretlersen IV ölçümü Google Drive'daki `abd_borsa/iv_gecmisi.csv` dosyasına eklenir (ilk seferde Drive izni ister). Zamanla kendi IV Rank geçmişimiz oluşur.

SECILEN_HISSE = secilen_hisse.strip().upper()
YON_SECIMI = yon_secimi
VADE_SECIMI = vade_secimi.strip()

In [ ]:
#@title ⚙️ Kurulum
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "yfinance", "py_vollib"],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("✅ yfinance ve py_vollib hazır.")

In [ ]:
#@title 🧰 Fonksiyonlar (dokunmana gerek yok)
import math, warnings, datetime as dt
import numpy as np
import pandas as pd
import requests
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import brentq
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

# py_vollib varsa onu kullan, yoksa aynı Black-Scholes hesabını kendimiz yaparız
try:
    from py_vollib.black_scholes.implied_volatility import implied_volatility as _vl_iv
    from py_vollib.black_scholes.greeks.analytical import delta as _vl_delta
    VOLLIB = True
except Exception:
    VOLLIB = False


# ---------- Yardımcılar ----------
def tr_sayi(x, ondalik=2):
    """1234.5 -> '1.234,50' (Türkçe biçim)"""
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "-"
    s = f"{x:,.{ondalik}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")

def tr_tarih(d):
    return pd.Timestamp(d).strftime("%d/%m/%Y")

def gun_kaldi(vade):
    return (pd.Timestamp(vade).date() - dt.date.today()).days

def risksiz_faiz():
    """13 haftalık ABD hazine bonosu faizi (^IRX); alınamazsa %4."""
    try:
        r = yf.Ticker("^IRX").history(period="5d")["Close"].dropna().iloc[-1] / 100
        if 0 < r < 0.2:
            return float(r)
    except Exception:
        pass
    return 0.04

def son_fiyat(tk):
    return float(tk.history(period="5d")["Close"].dropna().iloc[-1])


# ---------- Black-Scholes: örtük volatilite ve delta ----------
def _bs_fiyat(flag, S, K, T, r, s):
    d1 = (math.log(S / K) + (r + s * s / 2) * T) / (s * math.sqrt(T))
    d2 = d1 - s * math.sqrt(T)
    if flag == "c":
        return S * norm.cdf(d1) - K * math.exp(-r * T) * norm.cdf(d2)
    return K * math.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def ortuk_vol(fiyat, S, K, T, r, flag):
    if not (fiyat > 0 and S > 0 and K > 0 and T > 0):
        return np.nan
    iv = np.nan
    if VOLLIB:
        try:
            iv = float(_vl_iv(fiyat, S, K, T, r, flag))
        except Exception:
            iv = np.nan
    if not (iv == iv) or iv <= 0:
        try:
            iv = brentq(lambda s: _bs_fiyat(flag, S, K, T, r, s) - fiyat, 1e-4, 5.0)
        except Exception:
            iv = np.nan
    return iv if (iv == iv and 0.01 <= iv <= 5) else np.nan

def delta_hesap(flag, S, K, T, r, s):
    if not (s == s and s > 0 and T > 0):
        return np.nan
    if VOLLIB:
        try:
            return float(_vl_delta(flag, S, K, T, r, s))
        except Exception:
            pass
    d1 = (math.log(S / K) + (r + s * s / 2) * T) / (s * math.sqrt(T))
    return norm.cdf(d1) if flag == "c" else norm.cdf(d1) - 1


def vega_hesap(S, K, T, r, s):
    if not (s == s and s > 0 and T > 0):
        return 0.0
    d1 = (math.log(S / K) + (r + s * s / 2) * T) / (s * math.sqrt(T))
    return S * norm.pdf(d1) * math.sqrt(T)


# ---------- Opsiyon zinciri ----------
_ZINCIR_ONBELLEK = {}

def zincir(tk, vade, S, r):
    """Bir vadenin call ve put'larını tek tabloda, kendi IV ve delta hesabımızla döndürür."""
    anahtar = (tk.ticker, vade)
    if anahtar in _ZINCIR_ONBELLEK:
        return _ZINCIR_ONBELLEK[anahtar]
    oc = tk.option_chain(vade)
    T = max(gun_kaldi(vade), 1) / 365
    parcalar = []
    for flag, df in (("c", oc.calls), ("p", oc.puts)):
        d = df[["strike", "bid", "ask", "lastPrice", "volume", "openInterest", "impliedVolatility"]].copy()
        d["tip"] = flag
        parcalar.append(d)
    z = pd.concat(parcalar, ignore_index=True)
    for k in ["bid", "ask", "lastPrice", "volume", "openInterest"]:
        z[k] = pd.to_numeric(z[k], errors="coerce").fillna(0)
    canli = (z["bid"] > 0) & (z["ask"] > 0)
    z["fiyat"] = np.where(canli, (z["bid"] + z["ask"]) / 2, z["lastPrice"])
    z["fiyat_kaynagi"] = np.where(canli, "orta", "son işlem")
    z["makas_%"] = np.where(canli, (z["ask"] - z["bid"]) / z["fiyat"].where(z["fiyat"] > 0) * 100, np.nan)
    z["iv"] = [ortuk_vol(f, S, K, T, r, t) for f, K, t in zip(z["fiyat"], z["strike"], z["tip"])]
    yedek = z["impliedVolatility"].where(z["impliedVolatility"].between(0.01, 5))
    z["iv"] = z["iv"].fillna(yedek)
    z["delta"] = [delta_hesap(t, S, K, T, r, s) for t, K, s in zip(z["tip"], z["strike"], z["iv"])]
    z["likit"] = (z["openInterest"] >= MIN_ACIK_POZISYON) & (z["makas_%"].isna() | (z["makas_%"] <= MAX_MAKAS_YUZDE))
    z["vade"] = vade
    _ZINCIR_ONBELLEK[anahtar] = z
    return z

def uygun_vadeler(tk):
    return [v for v in tk.options if MIN_GUN <= gun_kaldi(v) <= MAX_GUN]

def atm_satirlari(z, S):
    """Fiyata en yakın strike'taki call ve put."""
    k = z.loc[(z["strike"] - S).abs().idxmin(), "strike"]
    c = z[(z["tip"] == "c") & (z["strike"] == k)].iloc[0]
    p = z[(z["tip"] == "p") & (z["strike"] == k)].iloc[0]
    return c, p

def atm_iv(z, S):
    c, p = atm_satirlari(z, S)
    return np.nanmean([c["iv"], p["iv"]])


# ---------- ADIM 1: Kalabalığın en çok konuştukları ----------
def apewisdom_liste(filtre, adet):
    j = requests.get(f"https://apewisdom.io/api/v1.0/filter/{filtre}/page/1", headers=UA, timeout=20).json()
    df = pd.DataFrame(j.get("results", []))
    if df.empty:
        return df
    for k in ["rank", "mentions", "upvotes", "rank_24h_ago", "mentions_24h_ago"]:
        if k in df:
            df[k] = pd.to_numeric(df[k], errors="coerce")
    df = df.head(adet)
    return pd.DataFrame({
        "hisse": df["ticker"].str.upper(),
        "reddit_sira": df["rank"],
        "reddit_bahsedilme": df["mentions"],
        "24s_once": df.get("mentions_24h_ago"),
    })

def stocktwits_trend():
    j = requests.get("https://api.stocktwits.com/api/2/trending/symbols.json", headers=UA, timeout=20).json()
    semboller = [s.get("symbol", "") for s in j.get("symbols", [])]
    return [s.upper() for s in semboller if s and "." not in s and "-" not in s]

def likidite_kontrol(sembol):
    sonuc = {"hisse": sembol, "fiyat": np.nan, "opsiyon": "yok", "zincir_OI": 0,
             "ATM_OI": 0, "ATM_makas_%": np.nan, "likit": False}
    try:
        tk = yf.Ticker(sembol)
        vadeler = uygun_vadeler(tk)
        if not vadeler:
            return sonuc
        S = son_fiyat(tk)
        v = min(vadeler, key=lambda x: abs(gun_kaldi(x) - 30))
        oc = tk.option_chain(v)
        z = pd.concat([oc.calls.assign(tip="c"), oc.puts.assign(tip="p")], ignore_index=True)
        z["openInterest"] = pd.to_numeric(z["openInterest"], errors="coerce").fillna(0)
        k = z.loc[(z["strike"] - S).abs().idxmin(), "strike"]
        atm = z[z["strike"] == k]
        canli = atm[(atm["bid"] > 0) & (atm["ask"] > 0)]
        makas = ((canli["ask"] - canli["bid"]) / ((canli["ask"] + canli["bid"]) / 2) * 100).mean() if len(canli) else np.nan
        sonuc.update(fiyat=round(S, 2), opsiyon="var", zincir_OI=int(z["openInterest"].sum()),
                     ATM_OI=int(atm["openInterest"].sum()), **{"ATM_makas_%": round(makas, 1) if makas == makas else np.nan})
        sonuc["likit"] = (sonuc["zincir_OI"] >= 10 * MIN_ACIK_POZISYON and sonuc["ATM_OI"] >= MIN_ACIK_POZISYON
                          and (makas != makas or makas <= MAX_MAKAS_YUZDE))
    except Exception as e:
        sonuc["opsiyon"] = f"hata: {str(e)[:40]}"
    return sonuc

def adim1_tarama():
    print("🔎 Reddit (ApeWisdom) ve StockTwits trend listeleri çekiliyor...")
    try:
        reddit = apewisdom_liste(REDDIT_FILTRESI, ADAY_SAYISI)
    except Exception as e:
        print(f"   ⚠️ ApeWisdom alınamadı: {e}")
        reddit = pd.DataFrame(columns=["hisse", "reddit_sira", "reddit_bahsedilme", "24s_once"])
    try:
        st = stocktwits_trend()
    except Exception as e:
        print(f"   ⚠️ StockTwits alınamadı: {e}")
        st = []
    tum = list(dict.fromkeys(list(reddit["hisse"]) + st[:ADAY_SAYISI]))
    tablo = pd.DataFrame({"hisse": tum}).merge(reddit, on="hisse", how="left")
    tablo["stocktwits_trend"] = tablo["hisse"].isin(st)
    tablo["iki_kaynakta"] = tablo["reddit_sira"].notna() & tablo["stocktwits_trend"]
    print(f"   {len(tablo)} aday bulundu, opsiyon likiditesi kontrol ediliyor...")
    lik = pd.DataFrame([likidite_kontrol(s) for s in tablo["hisse"]])
    tablo = tablo.merge(lik, on="hisse", how="left")
    tablo = tablo.sort_values(["likit", "iki_kaynakta", "reddit_bahsedilme"],
                              ascending=[False, False, False], na_position="last").reset_index(drop=True)
    return tablo


# ---------- ADIM 2: Yön ----------
def soz_sinyali(sembol, sayfa=3):
    bull = bear = toplam = 0
    imlec = None
    for _ in range(sayfa):
        url = f"https://api.stocktwits.com/api/2/streams/symbol/{sembol}.json"
        j = requests.get(url, params={"max": imlec} if imlec else None, headers=UA, timeout=20).json()
        mesajlar = j.get("messages", [])
        if not mesajlar:
            break
        for m in mesajlar:
            toplam += 1
            duygu = ((m.get("entities") or {}).get("sentiment") or {}).get("basic")
            bull += duygu == "Bullish"
            bear += duygu == "Bearish"
        imlec = mesajlar[-1]["id"]
    etiketli = bull + bear
    oran = bull / etiketli if etiketli else np.nan
    puan = 0
    if etiketli >= 10:
        puan = 1 if oran >= 0.6 else (-1 if oran <= 0.4 else 0)
    return {"mesaj": toplam, "boga": bull, "ayi": bear, "boga_orani": oran, "puan": puan}

def para_sinyali(tk, vadeler, S, r):
    zz = pd.concat([zincir(tk, v, S, r) for v in vadeler])
    hacim_c = zz.loc[zz["tip"] == "c", "volume"].sum(); hacim_p = zz.loc[zz["tip"] == "p", "volume"].sum()
    oi_c = zz.loc[zz["tip"] == "c", "openInterest"].sum(); oi_p = zz.loc[zz["tip"] == "p", "openInterest"].sum()
    pc_hacim = hacim_p / hacim_c if hacim_c else np.nan
    pc_oi = oi_p / oi_c if oi_c else np.nan
    # Çarpıklık: 30 güne en yakın vadede 25 deltalık put IV - 25 deltalık call IV
    v30 = min(vadeler, key=lambda x: abs(gun_kaldi(x) - 30))
    z = zincir(tk, v30, S, r).dropna(subset=["delta", "iv"])
    try:
        p25 = z[z["tip"] == "p"].iloc[(z[z["tip"] == "p"]["delta"] + 0.25).abs().argsort()[:1]]["iv"].iloc[0]
        c25 = z[z["tip"] == "c"].iloc[(z[z["tip"] == "c"]["delta"] - 0.25).abs().argsort()[:1]]["iv"].iloc[0]
        carpiklik = (p25 - c25) * 100
    except Exception:
        carpiklik = np.nan
    olcut = pc_hacim if pc_hacim == pc_hacim and hacim_c + hacim_p > 0 else pc_oi
    puan = 1 if olcut < 0.7 else (-1 if olcut > 1.0 else 0)
    return {"pc_hacim": pc_hacim, "pc_oi": pc_oi, "carpiklik": carpiklik, "puan": puan,
            "hacim_toplam": int(hacim_c + hacim_p)}

def fiyat_sinyali(tk):
    h = tk.history(period="1y")["Close"].dropna()
    S = float(h.iloc[-1])
    sma20, sma50, sma200 = (float(h.rolling(n).mean().iloc[-1]) for n in (20, 50, 200))
    getiri20 = (S / float(h.iloc[-21]) - 1) * 100
    hv20 = float(np.log(h).diff().tail(20).std() * math.sqrt(252))
    if S > sma20 > sma50:
        puan, aciklama = 1, "SMA20 ve SMA50 üstünde, kısa vadeli trend yukarı"
    elif S < sma20 < sma50:
        puan, aciklama = -1, "SMA20 ve SMA50 altında, kısa vadeli trend aşağı"
    else:
        puan, aciklama = 0, "ortalamalar arasında, trend karışık"
    return {"S": S, "sma20": sma20, "sma50": sma50, "sma200": sma200, "getiri20": getiri20,
            "hv20": hv20, "puan": puan, "aciklama": aciklama}


# ---------- ADIM 3: Vade ve strike ----------
def bilanco_tarihi(tk):
    bugun = dt.date.today()
    try:
        cal = tk.calendar
        if isinstance(cal, dict) and cal.get("Earnings Date"):
            gelecek = [pd.Timestamp(d).date() for d in cal["Earnings Date"] if pd.Timestamp(d).date() >= bugun]
            if gelecek:
                return min(gelecek)
    except Exception:
        pass
    try:
        ed = tk.get_earnings_dates(limit=8)
        gelecek = [d.date() for d in ed.index if d.date() >= bugun]
        if gelecek:
            return min(gelecek)
    except Exception:
        pass
    return None

def vade_tablosu(tk, vadeler, S, r, bilanco):
    satirlar = []
    for v in vadeler:
        z = zincir(tk, v, S, r)
        c, p = atm_satirlari(z, S)
        hareket = c["fiyat"] + p["fiyat"]
        satirlar.append({
            "vade": v, "gün": gun_kaldi(v), "ATM_IV_%": round(atm_iv(z, S) * 100, 1),
            "beklenen_hareket_$": round(hareket, 2), "beklenen_hareket_%": round(hareket / S * 100, 1),
            "bilançoyu_kapsıyor": bool(bilanco and bilanco <= pd.Timestamp(v).date()),
            "likit_kontrat": int(z["likit"].sum()),
        })
    return pd.DataFrame(satirlar)

def bilanco_hareketi(tk, S, r, bilanco):
    """Bilançodan sonraki ilk vadenin straddle fiyatı: piyasanın bilançoyu da içeren beklenen hareketi.
    Bu vade 7 gün kuralına takılmaz, sadece bilgi amaçlıdır."""
    if not bilanco:
        return None
    sonraki = [v for v in tk.options if pd.Timestamp(v).date() > bilanco]
    if not sonraki:
        return None
    v = sonraki[0]
    z = zincir(tk, v, S, r)
    c, p = atm_satirlari(z, S)
    hareket = c["fiyat"] + p["fiyat"]
    return {"vade": v, "gun": gun_kaldi(v), "hareket": hareket, "yuzde": hareket / S * 100}

def vade_sec(vt):
    if VADE_SECIMI:
        return VADE_SECIMI
    aday = vt[(~vt["bilançoyu_kapsıyor"]) & (vt["likit_kontrat"] > 0)]
    if aday.empty:
        aday = vt
    return aday.iloc[(aday["gün"] - 30).abs().argsort()].iloc[0]["vade"]

def kisa_bacak(z, tip):
    """Delta aralığındaki kontratlardan 0,25'e en yakın olanı (önce likit olanlar)."""
    hedef = (MIN_DELTA + MAX_DELTA) / 2
    d = z[(z["tip"] == tip) & z["delta"].abs().between(MIN_DELTA, MAX_DELTA)].copy()
    if d.empty:
        return None
    d["uzaklik"] = (d["delta"].abs() - hedef).abs()
    d = d.sort_values(["likit", "uzaklik"], ascending=[False, True])
    return d.iloc[0]

def atm_bacak(z, tip, S):
    d = z[z["tip"] == tip].copy()
    return d.iloc[(d["strike"] - S).abs().argsort()].iloc[0]

def kanat(z, tip, K, yon, S):
    """K'dan 'yon' (+1 yukarı, -1 aşağı) tarafta koruma bacağı.
    Genişlik hisse fiyatının MAX_GENISLIK_YUZDE'sini aşmaz; bu sınırda strike yoksa en dar seçenek alınır ve işaretlenir."""
    sinir = S * MAX_GENISLIK_YUZDE / 100
    d = z[(z["tip"] == tip)].copy()
    d["gen"] = (d["strike"] - K) * yon
    d = d[d["gen"] > 0]
    if d.empty:
        return None
    icinde = d[d["gen"] <= sinir + 1e-9]
    if not icinde.empty:
        # sınıra en yakın (en geniş) strike, önce likit olanlar
        return icinde.sort_values(["likit", "gen"], ascending=[False, False]).iloc[0]
    return d.sort_values("gen").iloc[0]

# ---------- ADIM 4: Strateji hesapları ----------
def kar_zarar(bacaklar, x):
    """bacaklar: [(adet, tip, strike, prim)] ; hisse başına vade sonu kâr/zarar."""
    toplam = np.zeros_like(x, dtype=float)
    for adet, tip, K, prim in bacaklar:
        ic = np.maximum(x - K, 0) if tip == "c" else np.maximum(K - x, 0)
        toplam += adet * (ic - prim)
    return toplam

def vega_yorumu(bacaklar, S, T, r, sigma):
    """Net vega işaretine göre IV düşüşünün yapıya etkisi; çok küçük net vega 'nötre yakın' sayılır."""
    vegalar = [a * vega_hesap(S, K, T, r, sigma) for a, _, K, _ in bacaklar]
    net, en_buyuk = sum(vegalar), max(abs(v) for v in vegalar)
    if en_buyuk == 0 or abs(net) < 0.15 * en_buyuk:
        return "nötre yakın"
    return "lehine" if net < 0 else "aleyhine"

def strateji_olc(ad, bacaklar, S, sigma, T, r, bilanco_kapsiyor=False):
    x = np.linspace(0, S * 3, 6001)
    kz = kar_zarar(bacaklar, x)
    net = -sum(a * p for a, _, _, p in bacaklar)  # + ise kredi, - ise borç (debit)
    sinirsiz = kz[-1] > kz[-2] + 1e-9
    maks_kar = np.inf if sinirsiz else kz.max()
    maks_zarar = kz.min()
    isaret = np.sign(kz)
    basabas = []
    for i in np.where(np.diff(isaret) != 0)[0]:
        if kz[i] == kz[i + 1]:
            continue
        basabas.append(x[i] - kz[i] * (x[i + 1] - x[i]) / (kz[i + 1] - kz[i]))
    # Kâr olasılığı: log-normal dağılım, ATM IV ile
    with np.errstate(divide="ignore"):
        zskor = (np.log(np.maximum(x, 1e-9) / S) - (r - sigma ** 2 / 2) * T) / (sigma * math.sqrt(T))
    cdf = norm.cdf(zskor)
    pop = float(np.sum(np.diff(cdf)[(kz[:-1] > 0) & (kz[1:] > 0)]))
    # Spread genişliği: aynı tipteki bacaklar arasındaki strike farkı
    genislikler = []
    for t in ("c", "p"):
        ks = sorted(K for a, tt, K, _ in bacaklar if tt == t)
        if len(ks) == 2:
            genislikler.append(ks[1] - ks[0])
    if genislikler:
        gy = max(genislikler) / S * 100
        gen_not = f"%{tr_sayi(gy, 1)}" + (" ⚠️ kural dışı" if gy > MAX_GENISLIK_YUZDE + 1e-9 else "")
    else:
        gen_not = "-"
    def bacak_yazi(a, t, K, p):
        return f"{'+' if a > 0 else '−'}1 {'Call' if t == 'c' else 'Put'} {tr_sayi(K)} @ {tr_sayi(p)}"
    return {
        "strateji": ad,
        "bacaklar": " | ".join(bacak_yazi(*b) for b in bacaklar),
        "tür": "Kredi" if net > 0 else "Borç (debit)",
        "genişlik": gen_not,
        "net_prim_$": round(abs(net) * 100, 0),
        "maks_kâr_$": "sınırsız" if sinirsiz else round(maks_kar * 100, 0),
        "maks_zarar_$": round(maks_zarar * 100, 0),
        "başabaş": " / ".join(tr_sayi(b) for b in basabas),
        "kâr_olasılığı_%": round(pop * 100, 0),
        "getiri/risk": "-" if sinirsiz or maks_zarar == 0 else round(maks_kar / abs(maks_zarar), 2),
        **({"IV_düşüşü": vega_yorumu(bacaklar, S, T, r, sigma)} if bilanco_kapsiyor else {}),
        "_bacaklar": bacaklar,
    }

def stratejiler_kur(z, S):
    def b(satir, adet):
        return (adet, satir["tip"], float(satir["strike"]), float(satir["fiyat"]))
    ac, ap = atm_bacak(z, "c", S), atm_bacak(z, "p", S)
    kc, kp = kisa_bacak(z, "c"), kisa_bacak(z, "p")
    liste = {"Long call": [b(ac, 1)], "Long put": [b(ap, 1)], "Long straddle": [b(ac, 1), b(ap, 1)]}
    # Not: iki bacaklı yapılarda genişlik kuralı kontrolü strateji_olc içinde yapılır
    w = kanat(z, "c", ac["strike"], +1, S)
    if w is not None: liste["Bull call (debit)"] = [b(ac, 1), b(w, -1)]
    w = kanat(z, "p", ap["strike"], -1, S)
    if w is not None: liste["Bear put (debit)"] = [b(ap, 1), b(w, -1)]
    if kp is not None:
        w = kanat(z, "p", kp["strike"], -1, S)
        if w is not None: liste["Bull put (kredi)"] = [b(kp, -1), b(w, 1)]
    if kc is not None:
        w = kanat(z, "c", kc["strike"], +1, S)
        if w is not None: liste["Bear call (kredi)"] = [b(kc, -1), b(w, 1)]
    if "Bull put (kredi)" in liste and "Bear call (kredi)" in liste:
        liste["Iron condor (kredi)"] = liste["Bull put (kredi)"] + liste["Bear call (kredi)"]
    return liste

UYUM = {
    ("Yükseliş", "ucuz"): ["Long call", "Bull call (debit)"],
    ("Yükseliş", "normal"): ["Bull call (debit)", "Bull put (kredi)"],
    ("Yükseliş", "pahalı"): ["Bull put (kredi)"],
    ("Düşüş", "ucuz"): ["Long put", "Bear put (debit)"],
    ("Düşüş", "normal"): ["Bear put (debit)", "Bear call (kredi)"],
    ("Düşüş", "pahalı"): ["Bear call (kredi)"],
    ("Nötr", "ucuz"): ["Long straddle"],
    ("Nötr", "normal"): ["Iron condor (kredi)"],
    ("Nötr", "pahalı"): ["Iron condor (kredi)"],
}

def iv_rejimi(iv, hv, ivr=-1):
    """Önce IV rank (girildiyse), yoksa IV/HV oranıyla opsiyonların pahalı mı ucuz mu olduğuna karar verir."""
    oran = iv / hv if hv else np.nan
    if oran != oran:
        rejim_hv = "normal"
    else:
        rejim_hv = "pahalı" if oran >= 1.25 else ("ucuz" if oran <= 0.9 else "normal")
    if ivr is not None and ivr >= 0:
        rejim_ivr = "pahalı" if ivr >= IVR_PAHALI else ("ucuz" if ivr <= IVR_UCUZ else "normal")
        return oran, rejim_hv, rejim_ivr, "IV rank"
    return oran, rejim_hv, rejim_hv, "IV/HV"

def grafik(sembol, secilenler, S, hareket, vade):
    if not secilenler:
        return
    x = np.linspace(S * 0.7, S * 1.3, 600)
    plt.figure(figsize=(10, 5))
    for s in secilenler:
        plt.plot(x, kar_zarar(s["_bacaklar"], x) * 100, label=s["strateji"], linewidth=2)
    plt.axhline(0, color="gray", linewidth=0.8)
    plt.axvline(S, color="black", linestyle="--", linewidth=1, label=f"Fiyat {tr_sayi(S)}")
    plt.axvspan(S - hareket, S + hareket, color="gold", alpha=0.15, label="Beklenen hareket aralığı")
    plt.title(f"{sembol} · {tr_tarih(vade)} vadesi · vade sonu kâr/zarar (1 kontrat, $)")
    plt.xlabel("Vade sonunda hisse fiyatı"); plt.ylabel("Kâr / zarar ($)")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print("✅ Fonksiyonlar yüklendi." + ("" if VOLLIB else " (py_vollib yüklenemedi, aynı hesap yedek kodla yapılacak)"))


## Adımlar

In [ ]:
#@title 1️⃣ Hisse: kalabalık en çok neyi konuşuyor?
tarama = adim1_tarama()
display(tarama)
if SECILEN_HISSE:
    hisse = SECILEN_HISSE
else:
    likitler = tarama[tarama["likit"] == True]
    hisse = likitler.iloc[0]["hisse"] if len(likitler) else None
print(f"\n👉 Seçilen hisse: {hisse}" if hisse else "\n⚠️ Likit aday bulunamadı. Ayarlardan bir hisse yaz.")

In [ ]:
#@title 2️⃣ Yön: söz, para ve fiyat ne diyor?
assert hisse, "Hisse seçilmedi. Ayarlardaki secilen_hisse kutusuna bir hisse yaz."
tk = yf.Ticker(hisse)
r = risksiz_faiz()
fs = fiyat_sinyali(tk)
S = fs["S"]
vadeler = uygun_vadeler(tk)
assert vadeler, f"{hisse} için {MIN_GUN}-{MAX_GUN} gün arasında vade yok."
try:
    ss = soz_sinyali(hisse)
except Exception as e:
    print(f"⚠️ StockTwits alınamadı: {e}")
    ss = {"mesaj": 0, "boga": 0, "ayi": 0, "boga_orani": np.nan, "puan": 0}
ps = para_sinyali(tk, vadeler, S, r)

isaret = {1: "🟢 yükseliş", 0: "⚪ nötr", -1: "🔴 düşüş"}
sinyaller = pd.DataFrame([
    {"sinyal": "Söz (StockTwits)", "ölçüm": f"%{tr_sayi(ss['boga_orani'] * 100, 0)} boğa ({ss['boga']} boğa / {ss['ayi']} ayı, {ss['mesaj']} mesaj)", "yön": isaret[ss["puan"]]},
    {"sinyal": "Para (opsiyon akışı)", "ölçüm": f"put/call hacim {tr_sayi(ps['pc_hacim'])}, açık poz. {tr_sayi(ps['pc_oi'])}, çarpıklık {tr_sayi(ps['carpiklik'], 1)} puan", "yön": isaret[ps["puan"]]},
    {"sinyal": "Fiyat (trend)", "ölçüm": f"{tr_sayi(S)} $; {fs['aciklama']}; 20 günde %{tr_sayi(fs['getiri20'], 1)}", "yön": isaret[fs["puan"]]},
])
display(sinyaller)

toplam_puan = ss["puan"] + ps["puan"] + fs["puan"]
otomatik_yon = "Yükseliş" if toplam_puan >= 2 else ("Düşüş" if toplam_puan <= -2 else "Nötr")
yon = otomatik_yon if YON_SECIMI == "otomatik" else YON_SECIMI
print(f"Toplam puan: {toplam_puan:+d} → sinyallerin yönü: {otomatik_yon}")
if yon != otomatik_yon:
    print(f"Senin seçimin: {yon} (hesaplar buna göre yapılacak)")
if ss["puan"] and ps["puan"] == -ss["puan"]:
    print("⚠️ Söz ile para ayrışıyor: kalabalığın söylediği ile opsiyon akışı ters yönde.")
if ss["boga_orani"] == ss["boga_orani"] and ss["boga_orani"] >= 0.8:
    print("⚠️ Kalabalık aşırı iyimser; aşırı uçlar bazen ters sinyal olarak okunur.")
print("Not: Çarpıklık, 25 deltalık put IV'sinin 25 deltalık call IV'sinden kaç puan yüksek olduğunu gösterir. Yüksek değer, aşağı yönlü korumaya talebin fazla olduğu anlamına gelir.")

In [ ]:
#@title 3️⃣ Vade ve strike
bilanco = bilanco_tarihi(tk)
print(f"Bilanço tarihi: {tr_tarih(bilanco) if bilanco else 'bulunamadı'}")
vt = vade_tablosu(tk, vadeler, S, r, bilanco)
bh = bilanco_hareketi(tk, S, r, bilanco)
if bh:
    print(f"📣 Bilanço sonrası ilk vade {tr_tarih(bh['vade'])} ({bh['gun']} gün): piyasa ±{tr_sayi(bh['hareket'])} $ (%{tr_sayi(bh['yuzde'], 1)}) hareket fiyatlıyor. Bu vade yalnızca bilgi amaçlıdır, vade kuralına dahil değildir.")
display(vt)
vade = vade_sec(vt)
z = zincir(tk, vade, S, r)
T = max(gun_kaldi(vade), 1) / 365
bilanco_kapsiyor = bool(bilanco and bilanco <= pd.Timestamp(vade).date())
c_atm, p_atm = atm_satirlari(z, S)
hareket = c_atm["fiyat"] + p_atm["fiyat"]
print(f"\n👉 Seçilen vade: {tr_tarih(vade)} ({gun_kaldi(vade)} gün). Beklenen hareket: ±{tr_sayi(hareket)} $ (%{tr_sayi(hareket / S * 100, 1)}) → {tr_sayi(S - hareket)} – {tr_sayi(S + hareket)}")
if (z["fiyat_kaynagi"] == "son işlem").mean() > 0.5:
    print("ℹ️ Piyasa kapalı ya da alış-satış verisi yok: son işlem fiyatları kullanıldı, makas filtresi uygulanamadı.")

kolonlar = ["tip", "strike", "fiyat", "fiyat_kaynagi", "makas_%", "openInterest", "volume", "iv", "delta", "likit"]
adaylar = z[z["delta"].abs().between(MIN_DELTA, MAX_DELTA)][kolonlar].copy()
adaylar["tip"] = adaylar["tip"].map({"c": "Call", "p": "Put"})
adaylar["iv"] = (adaylar["iv"] * 100).round(1)
adaylar["delta"] = adaylar["delta"].round(2)
adaylar["makas_%"] = adaylar["makas_%"].round(1)
adaylar["fiyat"] = adaylar["fiyat"].round(2)
print(f"\nKısa bacak adayları (|delta| {tr_sayi(MIN_DELTA)}–{tr_sayi(MAX_DELTA)}):")
display(adaylar.sort_values(["tip", "strike"]).reset_index(drop=True))

In [ ]:
#@title 4️⃣ Yapı: debit mi, kredi mi, çok bacaklı mı?
iv_rank = -1 #@param {type:"number"}
#@markdown MarketChameleon'daki IV rank değeri (0–100). Bilmiyorsan -1 bırak, IV/HV oranı kullanılır. Değeri girdikten sonra sadece bu hücreyi yeniden çalıştırman yeterli.
iv_rank_turu = "IV30 % Rank (yüzdelik)" #@param ["IV30 % Rank (yüzdelik)", "52 hafta pozisyonu (klasik IV Rank)"]
sigma = atm_iv(z, S)
iv_hv, rejim_hv, rejim, rejim_kaynagi = iv_rejimi(sigma, fs["hv20"], iv_rank)
print(f"ATM IV: %{tr_sayi(sigma * 100, 1)} | Gerçekleşen volatilite (20 gün): %{tr_sayi(fs['hv20'] * 100, 1)} | IV/HV: {tr_sayi(iv_hv)} → {rejim_hv}")
if iv_rank >= 0:
    print(f"IV rank ({iv_rank_turu}): {tr_sayi(iv_rank, 0)} → {rejim}")
    if rejim != rejim_hv:
        print(f"⚠️ İki ölçüt ayrışıyor: IV rank '{rejim}', IV/HV '{rejim_hv}' diyor. Karar IV rank'e göre verildi.")
else:
    print(f"ℹ️ IV rank girilmedi, karar IV/HV oranına göre verildi. {hisse} için MarketChameleon'daki değeri yukarıdaki kutuya yazıp bu hücreyi yeniden çalıştırabilirsin.")
print(f"👉 Opsiyonlar: {rejim.upper()} (ölçüt: {rejim_kaynagi})")
if bilanco_kapsiyor:
    print("⚠️ Seçilen vade bilançoyu kapsıyor: IV bilanço beklentisiyle şişmiş olabilir, bilanço sonrası düşmesi (IV crush) beklenir.")
    print("   Tablodaki 'IV_düşüşü' sütunu, IV düştüğünde yapının bundan kazanıp (lehine) kaybettiğini (aleyhine) gösterir. Fiyatın bir gecede sıçrama riski ayrıca vardır.")

olcumler = [strateji_olc(ad, b, S, sigma, T, r, bilanco_kapsiyor) for ad, b in stratejiler_kur(z, S).items()]
uyumlu = UYUM[(yon, rejim)]
tablo = pd.DataFrame([{k: v for k, v in o.items() if k != "_bacaklar"} for o in olcumler])
tablo.insert(0, "uyum", tablo["strateji"].isin(uyumlu).map({True: "✅", False: ""}))
tablo = tablo.sort_values("uyum", ascending=False).reset_index(drop=True)
print(f"\nYön: {yon} · Opsiyonlar: {rejim} → analizin işaret ettiği yapı(lar): {', '.join(uyumlu)}")
display(tablo)
grafik(hisse, [o for o in olcumler if o["strateji"] in uyumlu], S, hareket, vade)

if iv_kaydet:
    try:
        from google.colab import drive
        import os
        drive.mount("/content/drive", force_remount=False)
        klasor = "/content/drive/MyDrive/abd_borsa"; os.makedirs(klasor, exist_ok=True)
        yol = f"{klasor}/iv_gecmisi.csv"
        satir = pd.DataFrame([{"tarih": dt.date.today().isoformat(), "hisse": hisse, "vade": vade,
                               "gun": gun_kaldi(vade), "fiyat": round(S, 2), "atm_iv": round(sigma, 4),
                               "hv20": round(fs["hv20"], 4)}])
        satir.to_csv(yol, mode="a", header=not os.path.exists(yol), index=False)
        print(f"💾 IV kaydedildi: {yol}")
        gecmis = pd.read_csv(yol)
        gecmis = gecmis[gecmis["hisse"] == hisse].drop_duplicates("tarih", keep="last")
        if len(gecmis) >= 20:
            yuzdelik = (gecmis["atm_iv"] < sigma).mean() * 100
            print(f"📈 Kendi kaydımıza göre IV yüzdeliği: {tr_sayi(yuzdelik, 0)} ({len(gecmis)} günlük kayıt)")
        else:
            print(f"   Kendi IV yüzdeliğimiz için {hisse} kaydı: {len(gecmis)}/20 gün")
    except Exception as e:
        print(f"⚠️ IV kaydedilemedi: {e}")

bilanco_satiri = ""
if bilanco_kapsiyor:
    bilanco_satiri = f"\nBilanço {tr_tarih(bilanco)[:5]}, vadenin içinde" + (f"; bilanço vadesi ±%{tr_sayi(bh['yuzde'], 1)}" if bh else "")
yon_satiri = f"Yön: {otomatik_yon} ({toplam_puan:+d})" + (f", seçilen yön: {yon}" if yon != otomatik_yon else "")
kart = f"""${hisse} · {tr_tarih(dt.date.today())}
Söz %{tr_sayi(ss['boga_orani'] * 100, 0)} boğa | Put/call {tr_sayi(ps['pc_hacim'])} | Fiyat {fs['aciklama'].split(',')[0].replace('SMA20 ve SMA50', 'SMA20/50')}
{yon_satiri}
Vade {tr_tarih(vade)[:5]} ({gun_kaldi(vade)} gün), beklenen hareket ±%{tr_sayi(hareket / S * 100, 1)}{bilanco_satiri}
{('IV rank ' + tr_sayi(iv_rank, 0)) if iv_rank >= 0 else ('IV/HV ' + tr_sayi(iv_hv))}: opsiyonlar {rejim}
Yapı: {', '.join(uyumlu)}
Öğrenme amaçlıdır, yatırım tavsiyesi değildir."""
print("\n" + "=" * 60 + "\nKOPYALAMAK İÇİN DÜZ METİN (X)\n" + "=" * 60)
print(kart)
print(f"\n({len(kart)} karakter)")